

### Завдання 1: Виклик LLM з базовим промптом

**Мета:** навчитися викликати LLM через LangChain зі звичайним текстовим промптом.

**Що потрібно зробити:**

1. Створіть промпт, який дозволяє отримати інформацію простою мовою на тему "Квантові обчислення". Відповідь моделі повинна містити визначення, ключові переваги та поточні дослідження в цій галузі.

2. Обмежте відповідь до 200 символів і пропишіть в промпті, аби відповідь була короткою (це зекономить вам час і гроші на згенеровані токени).

3. Встановіть своє значення температури на власний розсуд (тут немає правильного чи неправильного значення) і напишіть коментарем, чому ви обрали саме таке значення для цього завдання.

**Вибір моделі:** можна скористатись як моделлю з HuggingFace, так і ChatGPT будь-якої версії, яка вам до вподоби і пасує за прайсингом. В обох випадках потрібно імпортувати відповідний клас з LangChain для виклику LLM за API.

**Мова запитів:** промпти можна писати як українською, так і англійською — орієнтуйтесь на те, де і для чого ви хочете потім використовувати цей проєкт. У розв'язках промпти — українською.

---

**🔐 Як безпечно зберігати і підвантажувати API-ключі**

API-токен потрібно зчитувати з безпечного джерела, а **не хардкодити в ноутбуці**. Якщо хтось отримає доступ до вашого ключа, він буде витрачати токени за ваш рахунок, а вам це не треба :)

Є кілька способів. Перший ми використовували на лекції, ще два для розширення вашого розуміння, як ще це можна зробити і що шлях не лише один. Для виконання цього ДЗ можете використовувати будь-який спосіб підвантаження ключів у ноутбук.

**Спосіб 1: Файл `creds.json` (рекомендований)**

Створіть файл `creds.json` з вашими ключами, завантажте його в Google Colab під час роботи, але **не здавайте** цей файл у ДЗ і **не комітьте** в git.

```python
import json
with open("creds.json") as f:
    creds = json.load(f)
api_key = creds["HF_TOKEN"]
```

**Спосіб 2: Google Colab Secrets**

У лівій панелі Colab натисніть іконку 🔑 (Secrets) → "Add new secret" → введіть назву (наприклад, `HF_TOKEN`) та значення ключа → увімкніть тогл доступу для ноутбука.

```python
from google.colab import userdata
api_key = userdata.get("HF_TOKEN")
```

Зручно тим, що ключ зберігається в акаунті і доступний у всіх ваших ноутбуках. Мінус — при кожній новій сесії потрібно перевірити, що доступ увімкнено.

**Спосіб 3: Google AI Studio (для Gemini API)**

Якщо працюєте з моделями Google Gemini, отримати безкоштовний API-ключ можна в [Google AI Studio](https://aistudio.google.com/app/apikey): увійдіть з Google-акаунтом → натисніть "Get API key" → "Create API key". Далі використовуйте ключ через будь-який із способів вище.



In [1]:
!pip install -q langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 16.6 MB/s eta 0:00:00


In [2]:
import json
import os
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

with open('creds.json') as file:
    creds = json.load(file)

os.environ["HUGGINGFACEHUB_API_TOKEN"] = creds["HUGGINGFACEHUB_API_TOKEN"]

llm_hf = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    max_new_tokens=200,
    temperature=0.1,
)

chat_model = ChatHuggingFace(llm=llm_hf)

prompt = """
Поясни тему 'Квантові обчислення' коротко (до 200 символів).
Включи: визначення, переваги, дослідження.
"""

response = chat_model.invoke(prompt)
print(f"Відповідь:\n{response.content}")

Відповідь:
Квантові обчислення — це технологія, яка використовує квантові ефекти для обчислень, змушуючи комп'ютери розв'язувати складні задачі швидше, ніж традиційні комп'ютери. Переваги: швидкість обчислень, ефективність для криптографії. Дослідження фокусуються на стабільності квантових станив та розробці квантових алгоритмів.


Температура 0.1 для точності, що важливо для наукового визначення

### Завдання 2: Створення параметризованого промпта для генерації тексту
Тепер ми хочемо оновити попередній фукнціонал так, аби в промпт ми могли передавати тему як параметр. Для цього скористайтесь `PromptTemplate` з `langchain` і реалізуйте параметризований промпт та виклик моделі з ним.

Запустіть оновлений функціонал (промпт + модел) для пояснень про теми
- "Баєсівські методи в машинному навчанні"
- "Трансформери в машинному навчанні"
- "Explainable AI"

Виведіть результати відпрацювання моделі на екран.

In [3]:
from langchain_core.prompts import PromptTemplate

template = """
Ти — науковий асистент. Поясни тему "{topic}" простою мовою.
Відповідь має бути короткою (до 250 символів) та містити головну суть.
"""

prompt_template = PromptTemplate(
    input_variables=["topic"],
    template=template,
)

topics = [
    "Баєсівські методи в машинному навчанні",
    "Трансформери в машинному навчанні",
    "Explainable AI"
]

print("Генерація відповідей для тем:")
for t in topics:
    formatted_prompt = prompt_template.format(topic=t)
    res = chat_model.invoke(formatted_prompt)

    print(f"\nТема: {t}")
    print(res.content.strip())

Генерація відповідей для тем:

Тема: Баєсівські методи в машинному навчанні
Баєсівські методи в машинному навчанні використовують теорію ймовірностей для підсумовування доказів та обчислення ймовірностей випадків. Це допомагає у точнішому прогнозуванні результатів.

Тема: Трансформери в машинному навчанні
Трансформери — це алгоритми для обробки тексту, які можуть розуміти порядок слів. Вони використовуються у машинному навчанні для перекладу, синтаксичного аналізу та інших задач.

Тема: Explainable AI
Explainable AI — це технологія, яка дозволяє розуміти та пояснювати рішення інтелектуальних систем, щоб люди могли зрозуміти, як та чому вона приймає визначені вирішення.




### Завдання 3: Використання агента для автоматизації процесів
Створіть агента, який допоможе автоматично шукати інформацію про останні наукові публікації в різних галузях. Наприклад, агент має знайти 5 останніх публікацій на тему штучного інтелекту.

**Кроки:**
1. Налаштуйте агента типу ReAct в LangChain для виконання автоматичних запитів.
2. Створіть промпт, який спрямовує агента шукати інформацію в інтернеті або в базах даних наукових публікацій.
3. Агент повинен видати список публікацій, кожна з яких містить назву, авторів і короткий опис.

Для взаємодії з пошуком там необхідно створити `Tool`. В лекції ми використовували `serpapi`. Можна продовжити користуватись ним, або обрати інше АРІ для пошуку (вони в тому числі є безкоштовні). Перелік різних АРІ, доступних в langchain, і орієнтир по вартості запитів можна знайти в окремому документі [тут](https://hannapylieva.notion.site/API-12994835849480a69b2adf2b8441cbb3?pvs=4).

Лишаю також нижче приклад використання одного з безкоштовних пошукових АРІ - DuckDuckGo (не потребує створення токена!)  - можливо він вам сподобається :)


In [7]:
!pip install -qU langchain langchain-classic langchain-community langchain-openai arxiv

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 6.5 MB/s eta 0:00:00


In [8]:
import os
import arxiv

from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool
from langchain_core.prompts import PromptTemplate
from langchain_classic.agents import AgentExecutor, create_react_agent

os.environ["OPENAI_API_KEY"] = creds["OPENAI_API_KEY"]

def search_latest_arxiv_publications(query: str) -> str:
    """
    Шукає 5 останніх публікацій в arXiv за заданою темою.
    Повертає назву, авторів, дату публікації та короткий опис.
    """

    client = arxiv.Client()

    search = arxiv.Search(
        query=query,
        max_results=5,
        sort_by=arxiv.SortCriterion.SubmittedDate,
        sort_order=arxiv.SortOrder.Descending
    )

    results = []

    for i, paper in enumerate(client.results(search), start=1):
        authors = ", ".join(author.name for author in paper.authors[:5])

        if len(paper.authors) > 5:
            authors += " та ін."

        summary = " ".join(paper.summary.split())
        short_summary = summary[:700] + "..." if len(summary) > 700 else summary

        results.append(
            f"{i}. Назва: {paper.title}\n"
            f"Автори: {authors}\n"
            f"Дата публікації: {paper.published.date()}\n"
            f"Короткий опис: {short_summary}\n"
            f"Посилання: {paper.entry_id}\n"
        )

    return "\n".join(results)

arxiv_tool = Tool(
    name="arxiv_latest_publications_search",
    func=search_latest_arxiv_publications,
    description=(
        "Використовуй цей інструмент для пошуку останніх наукових публікацій "
        "у базі arXiv. На вхід подається тема або ключові слова англійською мовою."
    )
)

tools = [arxiv_tool]

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

react_prompt = PromptTemplate.from_template("""
Ти науковий асистент, який допомагає знаходити останні наукові публікації.

Твоє завдання:
- знайти релевантні наукові публікації;
- використати доступний інструмент пошуку;
- надати відповідь українською мовою;
- для кожної публікації вказати назву, авторів і короткий опис.

Доступні інструменти:

{tools}

Використовуй такий формат:

Question: запит користувача
Thought: подумай, який інструмент треба використати
Action: назва інструмента, одна з [{tool_names}]
Action Input: пошуковий запит
Observation: результат роботи інструмента
Thought: я готовий сформувати фінальну відповідь
Final Answer: фінальна відповідь українською мовою

Question: {input}
Thought: {agent_scratchpad}
""")

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=react_prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

result = agent_executor.invoke({
    "input": "Знайди 5 останніх наукових публікацій на тему artificial intelligence."
})

print(result["output"])



> Entering new AgentExecutor chain...
Action: arxiv_latest_publications_search  
Action Input: artificial intelligence  1. Назва: Seeing Fast and Slow: Learning the Flow of Time in Videos
Автори: Yen-Siang Wu, Rundong Luo, Jingsen Zhu, Tao Tu, Ali Farhadi та ін.
Дата публікації: 2026-04-23
Короткий опис: How can we tell whether a video has been sped up or slowed down? How can we generate videos at different speeds? Although videos have been central to modern computer vision research, little attention has been paid to perceiving and controlling the passage of time. In this paper, we study time as a learnable visual concept and develop models for reasoning about and manipulating the flow of time in videos. We first exploit the multimodal cues and temporal structure naturally present in videos to learn, in a self-supervised manner, to detect speed changes and estimate playback speed. We then show that these learned temporal reasoning models enable us to curate the largest slow-motion vi



### Завдання 4: Створення агента-помічника для вирішення бізнес-задач

Створіть агента, який допомагає вирішувати задачі бізнес-аналітики. Агент має допомогти користувачу створити прогноз по продажам на наступний рік враховуючи рівень інфляції і погодні умови. Агент має вміти використовувати Python і ходити в інтернет аби отримати актуальні дані.

**Кроки:**
1. Налаштуйте агента, який працюватиме з аналітичними даними, заданими текстом. Користувач пише

```
Ми експортуємо апельсини з Бразилії. В 2022 експортували 200т, в 2023 - 190т, в 2024 - 210т, в 2025 - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2026 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
```

2. Створіть запит до агента, що містить чітке завдання – видати результат бізнес аналізу або написати, що він не може цього зробити і запит користувача (просто може бути все одним повідомлленням).

3. Запустіть агента і проаналізуйте результати. Що можна покращити?


In [9]:
!pip install -qU langchain-experimental langchain-tavily tavily-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 6.1 MB/s eta 0:00:00


In [10]:
from langchain_tavily import TavilySearch
from langchain_experimental.tools import PythonAstREPLTool
from langchain.agents import create_agent

os.environ["TAVILY_API_KEY"] = creds["TAVILY_API_KEY"]
os.environ["OPENAI_API_KEY"] = creds["OPENAI_API_KEY"]

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

search_tool = TavilySearch(
    max_results=5,
    topic="general"
)

python_tool = PythonAstREPLTool()

tools = [
    search_tool,
    python_tool
]

system_prompt = """
Ти бізнес-аналітик, який допомагає будувати прогноз експорту.

Ти маєш:
1. Проаналізувати історичні дані продажів або експорту.
2. Використати Python для числових розрахунків.
3. Використати інтернет-пошук для отримання актуальної інформації про:
   - погодні умови в Бразилії;
   - світовий попит на апельсини;
   - економічні фактори, зокрема інфляцію або споживчий попит.
4. Сформувати прогноз на наступний рік.
5. Чітко пояснити припущення.
6. Якщо даних недостатньо, прямо вказати обмеження прогнозу.

Відповідай українською мовою.

Формат відповіді:
- Вхідні дані
- Дані з інтернету
- Розрахунок у Python
- Прогноз
- Обмеження
- Що можна покращити
"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

user_query = """
Ми експортуємо апельсини з Бразилії.
В 2022 експортували 200 т, в 2023 - 190 т,
в 2024 - 210 т, в 2025 - 220 т.

Зроби оцінку, скільки ми зможемо експортувати апельсинів у 2026 році,
враховуючи погодні умови в Бразилії і попит на апельсини у світі
виходячи з економічної ситуації.

Якщо точний прогноз неможливий, побудуй обґрунтовану оцінку
і поясни всі припущення.
"""

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": user_query
            }
        ]
    }
)

print(result["messages"][-1].content)

### Прогноз
На основі розрахунків, прогнозований експорт апельсинів з Бразилії на 2026 рік становитиме приблизно **235 тонн**.

### Обмеження
1. **Невизначеність погодних умов**: Якщо в Бразилії відбудуться несприятливі погодні умови, це може суттєво знизити врожайність.
2. **Зміни в світовому попиті**: Якщо попит на апельсини зменшиться через економічні фактори або зміни в споживчих звичках, це може вплинути на експорт.
3. **Економічні коливання**: Інфляція, зміни в валютних курсах та інші економічні фактори можуть вплинути на ціни та попит.

### Що можна покращити
1. **Збір даних**: Регулярний моніторинг погодних умов та економічних показників може допомогти в точнішому прогнозуванні.
2. **Аналіз ринку**: Використання більш складних моделей прогнозування, таких як ARIMA або машинне навчання, може дати кращі результати.
3. **Дослідження споживчого попиту**: Вивчення тенденцій споживання на основі соціологічних опитувань може допомогти зрозуміти, як змінюється попит на апельсини.
